

# 📘 Guía de Implementación: Modelo Base SARSA con Gymnasium

**(Integrando el Notebook de Análisis de Datos)**

**Objetivo:** Crear una línea base rápida (solo con `SPY`) para validar la estrategia antes de pasar a Deep Learning.
**Origen de Datos:** Usaremos la lógica de descarga y limpieza definida en el *Notebook\_Descriptivo\_Dataset.ipynb*.

-----

## 1\. Configuración del Entorno

Primero, asegúrate de tener las librerías que usamos en el notebook más las de RL:

```bash
pip install gymnasium numpy pandas yfinance pandas-ta matplotlib
```

-----

## 2\. Obtención y Procesamiento de Datos (Basado en el Notebook)

No vamos a inventar datos. Vamos a usar el script del notebook para descargar el histórico real.

**Instrucciones para el script `data_loader.py`:**

1.  **Descarga:** Usa `yfinance` como en el notebook, pero para esta fase inicial **filtra solo los datos de 'SPY'** (queremos validar la lógica con un solo activo primero).
2.  **Limpieza:** Aplica el `fillna(method='ffill')` que vimos en el análisis para tapar los huecos de festivos.
3.  **Indicadores (`pandas-ta`):** Usa la librería del notebook para generar las señales técnicas.

Copia y pega este código para generar el Dataset de entrenamiento:

```python
import yfinance as yf
import pandas as pd
import pandas_ta as ta  # La librería del notebook

def get_sarsa_data():
    # 1. Descarga (Misma lógica del Notebook)
    # Usamos un rango amplio para que el agente vea diferentes ciclos
    df = yf.download("SPY", start="2010-01-01", end="2025-01-01", auto_adjust=False)
    
    # Aplanar MultiIndex si es necesario (yfinance a veces devuelve niveles extra)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # 2. Limpieza (Validado en el análisis de nulos del notebook)
    df = df.fillna(method='ffill').dropna()

    # 3. Cálculo de Indicadores (Feature Engineering)
    # Calculamos la SMA de 200 días para la Tendencia
    df['SMA_200'] = ta.sma(df['Close'], length=200)
    
    # Calculamos el RSI de 14 días para el Momento
    df['RSI'] = ta.rsi(df['Close'], length=14)
    
    # 4. Cálculo de Recompensa (Log Returns)
    # Usamos logaritmos para normalizar la volatilidad que vimos en los histogramas
    import numpy as np
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

    # Eliminamos los NaNs generados por los indicadores (los primeros 200 días)
    df = df.dropna()
    
    return df
```

-----

## 3\. Discretización: Convertir Datos en "Estados"

SARSA Tabular no puede leer "Precio: 450.32". Necesita leer "Estado 5". Vamos a convertir los indicadores continuos en cajas (bins).

**Añade esta función al pre-procesamiento:**

```python
def discretize_data(df):
    """
    Convierte los indicadores continuos en estados discretos para la Q-Table.
    """
    # --- Estado 1: Tendencia (0 o 1) ---
    # 1 (Alcista): Precio > Media 200
    # 0 (Bajista): Precio < Media 200
    df['State_Trend'] = np.where(df['Close'] > df['SMA_200'], 1, 0)

    # --- Estado 2: RSI (0, 1, 2) ---
    # 0 (Sobreventa): RSI < 30
    # 2 (Sobrecompra): RSI > 70
    # 1 (Neutral): Entre 30 y 70
    conditions = [
        (df['RSI'] < 30),
        (df['RSI'] > 70)
    ]
    choices = [0, 2]
    df['State_RSI'] = np.select(conditions, choices, default=1)
    
    return df
```

-----

## 4\. El Entorno de Gymnasium (`SarsaTradingEnv`)

Este es el "tablero de juego". El agente interactuará con esta clase.

```python
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class SarsaTradingEnv(gym.Env):
    def __init__(self, df):
        super(SarsaTradingEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        
        # --- ACCIONES (2) ---
        # 0: CASH (Estar fuera / Vender)
        # 1: LONG (Estar dentro / Comprar)
        self.action_space = spaces.Discrete(2)
        
        # --- ESTADOS (12) ---
        # Combinación de: 2 Tendencias x 3 RSIs x 2 Posiciones (Dentro/Fuera)
        self.observation_space = spaces.Discrete(12)
        
        self.current_step = 0
        self.in_position = 0 # 0 = Cash, 1 = Invested

    def _get_state_id(self):
        # Mapea la fila actual a un número único entre 0 y 11
        row = self.df.iloc[self.current_step]
        
        trend = int(row['State_Trend']) # 0 o 1
        rsi = int(row['State_RSI'])     # 0, 1 o 2
        pos = self.in_position          # 0 o 1
        
        # Fórmula de mapeo: (Tendencia * 6) + (RSI * 2) + Posicion
        state_id = (trend * 6) + (rsi * 2) + pos
        return state_id

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.in_position = 0 
        return self._get_state_id(), {}

    def step(self, action):
        # 1. Ejecutar Acción (Cambiar o mantener posición)
        self.in_position = action 
        
        # 2. Avanzar al día siguiente
        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        
        # 3. Calcular Recompensa
        # Solo obtenemos el retorno del mercado si estamos DENTRO (Action=1)
        # Si estamos en Cash (Action=0), el retorno es 0.
        market_return = self.df.iloc[self.current_step]['Log_Return']
        reward = market_return if self.in_position == 1 else 0
        
        # 4. Observar nuevo estado
        next_state = self._get_state_id()
        
        return next_state, reward, terminated, False, {}
```

-----

## 5\. El Bucle de Entrenamiento (SARSA)

Dile que copie este bucle. Es la implementación estándar de SARSA.

```python
# 1. Cargar y preparar datos del Notebook
df_raw = get_sarsa_data()
df_clean = discretize_data(df_raw)

# 2. Iniciar Entorno
env = SarsaTradingEnv(df_clean)

# 3. Inicializar Q-Table (12 estados x 2 acciones)
Q_table = np.zeros((env.observation_space.n, env.action_space.n))

# Parámetros (Ajustables)
alpha = 0.1      # Tasa de aprendizaje
gamma = 0.95     # Factor de descuento
epsilon = 1.0    # Exploración inicial
epsilon_decay = 0.995
min_epsilon = 0.01

# 4. Entrenar
for episode in range(500):
    state, _ = env.reset()
    
    # Epsilon-Greedy para la primera acción
    if np.random.rand() < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Q_table[state])
        
    done = False
    while not done:
        # Paso en el entorno
        next_state, reward, done, _, _ = env.step(action)
        
        # Elegir siguiente acción (SARSA requiere saber A' antes de actualizar)
        if np.random.rand() < epsilon:
            next_action = env.action_space.sample()
        else:
            next_action = np.argmax(Q_table[next_state])
            
        # Actualización SARSA
        # Q(s,a) = Q(s,a) + alpha * [R + gamma * Q(s',a') - Q(s,a)]
        target = reward + gamma * Q_table[next_state, next_action]
        Q_table[state, action] += alpha * (target - Q_table[state, action])
        
        # Avanzar
        state = next_state
        action = next_action
    
    # Decaer exploración
    epsilon = max(min_epsilon, epsilon * epsilon_decay)

print("Entrenamiento completado.")
print("Q-Table Final:\n", Q_table)
```

## 6\. Validación (Check Final)

Para saber si ha funcionado, dile que mire la **Q-Table** impresa al final.

  * Si en el estado donde `RSI=2` (Sobrecompra/Caro) y `Tendencia=0` (Bajista), el valor de la acción **0 (Vender)** es mayor que la acción **1 (Comprar)**, el modelo ha aprendido lógica financiera básica.